# CoDreamer

> CoDreamer (Toledo & Prorok, 2024): DreamerV3 whose agents communicate with graph attention (GATv2) inside the world model and in the actor-critic.

In [ ]:
#| default_exp wm.codreamer.core

In [ ]:
#| hide
from nbdev.showdoc import *

[CoDreamer](https://arxiv.org/abs/2406.13600) ("CoDreamer: Communication-Based Decentralised World
Models", Toledo and Prorok, 2024) extends DreamerV3 run per agent (with shared weights, here
`stable_marl.wm.dreamer`) with communication at two levels, both with
[GATv2](https://arxiv.org/abs/2105.14491) graph attention over the agents:

* **in the world model**: a GNN layer in the RSSM (here over the agents' recurrent states `h`, whose
  output enters the posterior and the prior of the stochastic state `z`) and in the reward and
  continuation heads; the decoder stays per agent;
* **in the actor-critic**: GNN torsos of the actor and the critic, over each agent's `[z ‖ h]`.

Each GNN layer concatenates its input to its output (the paper: against over-smoothing). The graph
links the agents within the communication range (`comm_range`, here the Chebyshev distance on the
grid), with their distance as edge features; in imagination, each trajectory keeps the graph of
the step it started from (the paper). Agents that finished leave the graph.

Who hears whom can also come from comm-core networks (`links`, `NetworkLinks`): every step, two
rounds of messages, the recurrent states `h` (for the RSSM's layer) then the features `[z ‖ h]`
(for the actor's), of their real sizes; what arrived is recorded in the replay, so that training
uses the graphs of acting. The reward, continuation and critic layers, used only in training, use
the second round's graph.

`CoDreamerModel` subclasses r2dreamer's `Dreamer` (the vendored files are unchanged): it replaces
the RSSM, the heads, the actor and the critic by communicating versions (`CommRSSM`, `GraphHead`)
that read the current graph from a shared `GraphContext`, and sets that graph before every use
(from the replayed batch, or from the transitions when acting).

In [ ]:
#| export
from __future__ import annotations

import copy
from collections import OrderedDict
from typing import Any, Sequence

import numpy as np
import torch
import torch.nn.functional as F
from fastcore.utils import patch
from omegaconf import DictConfig
from torch import nn
from torch.optim.lr_scheduler import LambdaLR

from stable_marl.wm.commnet.core import NetworkLinks, comm_mask
from stable_marl.wm.dreamer.core import DreamerAgents
from stable_marl.wm.dreamer.r2dreamer.dreamer import Dreamer
from stable_marl.wm.dreamer.r2dreamer.networks import LambdaLayer, MLPHead
from stable_marl.wm.dreamer.r2dreamer.optim import LaProp, clip_grad_agc_
from stable_marl.wm.dreamer.r2dreamer.rssm import RSSM
from stable_marl.wm.dreamer.r2dreamer.tools import weight_init_

### GATv2

`GATv2` (Brody et al., 2022) over the agents of each group: `e_ij = a · LeakyReLU(W_t x_i + W_s x_j
+ W_e d_ij)`, attention `α_ij = softmax_j e_ij` over the agents `j` that agent `i` hears (itself
always), output `Σ_j α_ij W_s x_j`, averaged over the heads.

In [ ]:
#| export
class GATv2(nn.Module):
    "GATv2 attention over the agents (dimension -2) with edge features."
    def __init__(self, in_dim: int, out_dim: int, edge_dim: int = 1, heads: int = 1, negative_slope: float = 0.2):
        super().__init__()
        self.heads, self.out_dim, self.negative_slope = heads, out_dim, negative_slope
        self.source = nn.Linear(in_dim, heads * out_dim)
        self.target = nn.Linear(in_dim, heads * out_dim)
        self.edge = nn.Linear(edge_dim, heads * out_dim, bias=False)
        self.attention = nn.Parameter(torch.empty(heads, out_dim))
        nn.init.xavier_uniform_(self.attention)

    def forward(
        self,
        x: torch.Tensor, # (..., N, in_dim)
        hears: torch.Tensor, # (..., N, N) bool: agent i hears agent j (its own: always)
        edges: torch.Tensor, # (..., N, N, edge_dim)
    ) -> torch.Tensor:
        "(..., N, out_dim)."
        N, H, C = x.shape[-2], self.heads, self.out_dim
        source = self.source(x).unflatten(-1, (H, C))                                   # (..., N, H, C)
        target = self.target(x).unflatten(-1, (H, C))
        e = target.unsqueeze(-3) + source.unsqueeze(-4) + self.edge(edges.to(x.dtype)).unflatten(-1, (H, C))
        score = (F.leaky_relu(e, self.negative_slope) * self.attention).sum(-1)         # (..., N_i, N_j, H)
        hears = hears.bool() | torch.eye(N, dtype=torch.bool, device=x.device)
        alpha = torch.softmax(score.masked_fill(~hears.unsqueeze(-1), float('-inf')), -2)
        return (alpha.unsqueeze(-1) * source.unsqueeze(-4)).sum(-3).mean(-2)            # (..., N, C)

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
torch.manual_seed(0)
gat = GATv2(4, 3, heads=2)
x, d = torch.randn(2, 3, 4), torch.rand(2, 3, 3, 1)
everyone, nobody = torch.ones(2, 3, 3, dtype=torch.bool), torch.zeros(2, 3, 3, dtype=torch.bool)
test_eq(gat(x, everyone, d).shape, (2, 3, 3))
other = x.clone(); other[:, 1] += 1
lost = everyone.clone(); lost[:, 0, 1] = False
test_close(gat(x, lost, d)[:, 0], gat(other, lost, d)[:, 0])                     # what agent 0 does not hear
assert not torch.allclose(gat(x, everyone, d)[:, 0], gat(other, everyone, d)[:, 0])
test_close(gat(x, nobody, d), gat.source(x).unflatten(-1, (2, 3)).mean(-2))      # alone: its own message

### The graph context

`GraphContext` holds the graph of the current use, for `N` agents per group (an env): for acting, one
graph per group (`(G, N, N)`, rows: group then agent); for training, one per group and step of the
replayed windows (`(G, T, N, N)`, rows `(G * N, T, ...)`), read one step at a time by the RSSM's
posterior rollout (`step`), and by the start step of every imagined trajectory (rows
`(G * N * T, ...)`). `message(gnn, x, round)` regroups `x` by agent, applies `gnn` with the graph of
`round` (`'rssm'` or `'policy'`), and returns its output in `x`'s layout.

In [ ]:
#| export
ROUNDS = ('rssm', 'policy')


class GraphContext:
    "The graph the communicating modules use now, for groups of `num_agents` agents."
    def __init__(self, num_agents: int):
        self.num_agents = num_agents
        self.hears: dict[str, torch.Tensor] = {}
        self.edges: torch.Tensor | None = None
        self.timed, self.step = False, None

    def set(self, hears: dict[str, torch.Tensor], edges: torch.Tensor, timed: bool):
        "`hears` per round and `edges` (`(G, N, N[, 1])`, or `(G, T, N, N[, 1])` with `timed`)."
        any_round = next(iter(hears.values()))
        edges = edges.unsqueeze(-1) if edges.dim() == any_round.dim() else edges     # (..., N, N, edge_dim)
        self.hears, self.edges, self.timed, self.step = hears, edges, timed, None

    def message(self, gnn: nn.Module, x: torch.Tensor, round: str) -> torch.Tensor:
        "`gnn` over the agents of each group, in `x`'s layout (last dimension: `gnn`'s output)."
        N, hears, edges = self.num_agents, self.hears[round], self.edges
        if not self.timed or self.step is not None:                          # rows: group, agent
            if self.step is not None:
                hears, edges = hears[:, self.step], edges[:, self.step]
            out = gnn(x.reshape(-1, N, x.shape[-1]), hears, edges)
            return out.reshape(*x.shape[:-1], out.shape[-1])
        G, T = hears.shape[:2]
        rest = x.shape[2:] if x.shape[0] == G * N else x.shape[1:]            # (G*N, T, ...) or (G*N*T, ...)
        grouped = x.reshape(G, N, T, *rest).movedim(1, -2)                    # (G, T, ..., N, F)
        extra = (1,) * (grouped.dim() - 4)
        out = gnn(grouped, hears.view(G, T, *extra, N, N), edges.view(G, T, *extra, N, N, -1))
        return out.movedim(-2, 1).reshape(*x.shape[:-1], out.shape[-1])

In [ ]:
#| hide
ctx = GraphContext(2)
G, T = 3, 4
hears = {r: torch.rand(G, T, 2, 2) > 0.5 for r in ROUNDS}
ctx.set(hears, torch.rand(G, T, 2, 2), timed=True)
x = torch.randn(G * 2, T, 4)
whole = ctx.message(gat, x, 'rssm')                                              # sequences (G*N, T, F)
flat = ctx.message(gat, x.reshape(G * 2 * T, 4), 'rssm').reshape(G * 2, T, -1)  # imagination starts (G*N*T, F)
test_close(whole, flat)
for t in range(T):                                                               # the posterior's steps
    ctx.step = t
    test_close(ctx.message(gat, x[:, t], 'rssm'), whole[:, t])
ctx.step = None
horizon = ctx.message(gat, x.reshape(G * 2 * T, 1, 4).expand(-1, 5, 4), 'rssm')  # (G*N*T, H, F): the start's graph
test_close(horizon[:, 2], flat.reshape(G * 2 * T, -1))

### The communicating modules

`CommRSSM` is r2dreamer's `RSSM` with a GATv2 layer over the recurrent states: the posterior and
the prior of `z` take `[h ‖ GAT(h)]` (their networks rebuilt for the wider input, as r2dreamer
builds them). `GraphHead` is an `MLPHead` (reward, continuation, actor, critic) on
`[feat ‖ GAT(feat)]`.

In [ ]:
#| export
def _logit_net(inp_dim: int, hidden: int, layers: int, stoch: int, discrete: int, act: str, name: str) -> nn.Sequential:
    "r2dreamer's posterior / prior network (Linear, RMSNorm, activation; logits reshaped to (stoch, discrete))."
    net = nn.Sequential()
    for i in range(layers):
        net.add_module(f"{name}_{i}", nn.Linear(inp_dim, hidden, bias=True))
        net.add_module(f"{name}_n_{i}", nn.RMSNorm(hidden, eps=1e-04, dtype=torch.float32))
        net.add_module(f"{name}_a_{i}", getattr(torch.nn, act)())
        inp_dim = hidden
    net.add_module(f"{name}_logit", nn.Linear(inp_dim, stoch * discrete))
    net.add_module(f"{name}_lambda", LambdaLayer(lambda x: x.reshape(*x.shape[:-1], stoch, discrete)))
    return net


class CommRSSM(RSSM):
    "r2dreamer's RSSM whose posterior and prior see the neighbours' recurrent states (GATv2)."
    def __init__(self, config, embed_size: int, act_dim: int, ctx: GraphContext, gnn_dim: int):
        super().__init__(config, embed_size, act_dim)
        self.ctx = ctx
        self.gnn = GATv2(self._deter, gnn_dim)
        self._obs_net = _logit_net(self._deter + gnn_dim + embed_size, self._hidden, self._obs_layers, self._stoch,
                                   self._discrete, config.act, 'obs_net')
        self._img_net = _logit_net(self._deter + gnn_dim, self._hidden, self._img_layers, self._stoch,
                                   self._discrete, config.act, 'img_net')
        self._obs_net.apply(weight_init_)
        self._img_net.apply(weight_init_)

    def communicate(self, deter: torch.Tensor) -> torch.Tensor:
        "`[h ‖ act(GAT(h))]`."
        return torch.cat([deter, F.silu(self.ctx.message(self.gnn, deter, 'rssm'))], -1)

    def observe(self, embed, action, initial, reset):
        "The posterior rollout, each step with its graph."
        try:
            stochs, deters, logits, (stoch, deter) = [], [], [], initial
            for i in range(action.shape[1]):
                self.ctx.step = i
                stoch, deter, logit = self.obs_step(stoch, deter, action[:, i], embed[:, i], reset[:, i])
                stochs.append(stoch); deters.append(deter); logits.append(logit)
        finally:
            self.ctx.step = None
        return torch.stack(stochs, 1), torch.stack(deters, 1), torch.stack(logits, 1)

    def obs_step(self, stoch, deter, prev_action, embed, reset):
        "r2dreamer's posterior step, the posterior seeing `[h ‖ GAT(h)]`."
        keep = lambda x: torch.where(reset.view(-1, *[1] * (x.dim() - 1)).bool(), torch.zeros_like(x), x)
        stoch, deter, prev_action = keep(stoch), keep(deter), keep(prev_action)
        deter = self._deter_net(stoch, deter, prev_action)
        logit = self._obs_net(torch.cat([self.communicate(deter), embed], dim=-1))
        return self.get_dist(logit).rsample(), deter, logit

    def prior(self, deter):
        "r2dreamer's prior, seeing `[h ‖ GAT(h)]`."
        logit = self._img_net(self.communicate(deter))
        return self.get_dist(logit).rsample(), logit


class GraphHead(nn.Module):
    "An `MLPHead` on `[feat ‖ act(GAT(feat))]`, with the graph of `round`."
    def __init__(self, config, feat_size: int, ctx: GraphContext, gnn_dim: int, round: str = 'policy'):
        super().__init__()
        self.ctx, self.round = ctx, round
        self.gnn = GATv2(feat_size, gnn_dim)
        self.head = MLPHead(config, feat_size + gnn_dim)

    def forward(self, feat):
        return self.head(torch.cat([feat, F.silu(self.ctx.message(self.gnn, feat, self.round))], -1))

### The model

`CoDreamerModel` builds r2dreamer's `Dreamer`, swaps in the communicating modules (and rebuilds its
optimizer over them, as `Dreamer.__init__` does), and sets the graph before acting (`act`) and
before each update (`update`, from the replayed batch's `hears`, `mask` and `position`). Every
communicating module, including the frozen copies `Dreamer` makes, shares its `GraphContext`.

In [ ]:
#| export
class CoDreamerModel(Dreamer):
    "r2dreamer's Dreamer with CoDreamer's communication (GATv2 in the RSSM, the reward and continuation heads, actor, critic)."
    def __init__(self, config: DictConfig, obs_space, act_space, num_agents: int, gnn_dim: int | None = None,
                 grid_size: int = 15):
        super().__init__(config, obs_space, act_space)
        if self.rep_loss == 'dreamerpro':
            raise NotImplementedError("CoDreamer runs with rep_loss 'dreamer', 'r2dreamer' or 'infonce'")
        self.ctx, self.grid_size = GraphContext(num_agents), grid_size
        gnn_dim = int(gnn_dim or config.actor.units)
        feat = self.rssm.feat_size
        self.rssm = CommRSSM(config.rssm, self.embed_size, self.act_dim, self.ctx, gnn_dim)
        self.reward = GraphHead(config.reward, feat, self.ctx, gnn_dim)
        self.cont = GraphHead(config.cont, feat, self.ctx, gnn_dim)
        self.actor = GraphHead(config.actor, feat, self.ctx, gnn_dim)
        self.value = GraphHead(config.critic, feat, self.ctx, gnn_dim)
        self._slow_value = copy.deepcopy(self.value)
        for param in self._slow_value.parameters():
            param.requires_grad = False
        self._optimize(config)
        self.clone_and_freeze()

    def _optimize(self, config):
        "As `Dreamer.__init__`: LaProp with warmup and adaptive gradient clipping, over the (new) modules."
        modules = {'rssm': self.rssm, 'actor': self.actor, 'value': self.value, 'reward': self.reward,
                   'cont': self.cont, 'encoder': self.encoder}
        modules.update({k: getattr(self, a) for k, a in (('decoder', 'decoder'), ('projector', 'prj')) if hasattr(self, a)})
        self._named_params = OrderedDict((f"{name}.{n}", p) for name, m in modules.items() for n, p in m.named_parameters())
        self._agc = lambda params: clip_grad_agc_(params, float(config.agc), float(config.pmin), foreach=True)
        self._optimizer = LaProp(self._named_params.values(), lr=config.lr, betas=(config.beta1, config.beta2), eps=config.eps)
        warmup = config.warmup
        self._scheduler = LambdaLR(self._optimizer, lr_lambda=lambda step: min(1.0, (step + 1) / warmup) if warmup else 1.0)

    def clone_and_freeze(self):
        "`Dreamer.clone_and_freeze`, with every copy sharing the graph context."
        super().clone_and_freeze()
        if getattr(self, 'ctx', None) is not None:
            for module in self.modules():
                if hasattr(module, 'ctx'):
                    module.ctx = self.ctx

    def set_graph(self, data: dict, timed: bool):
        """
        The graph of `data` (rows: group, then agent): `hears` (rows `(..., rounds, N)`: whose messages
        each agent got, each round), `mask` (agents in their episode) and `position`.
        """
        N = self.ctx.num_agents
        alive = data['mask'][..., 0] > 0                                      # (G*N[, T])
        position = data['position'].float()
        if timed:                                                             # (G*N, T, ...) -> (G, T, N, ...)
            alive = alive.reshape(-1, N, alive.shape[1]).transpose(1, 2)
            position = position.reshape(-1, N, *position.shape[1:]).transpose(1, 2)
            hears = data['hears'].reshape(-1, N, *data['hears'].shape[1:]).transpose(1, 2)   # (G, T, N, R, N)
        else:                                                                 # (G*N, ...) -> (G, N, ...)
            alive = alive.reshape(-1, N)
            position = position.reshape(-1, N, 2)
            hears = data['hears'].reshape(-1, N, *data['hears'].shape[1:])                     # (G, N, R, N)
        both = alive.unsqueeze(-1) & alive.unsqueeze(-2)
        rounds = {r: hears[..., k, :].bool() & both for k, r in enumerate(ROUNDS)}
        edges = (position.unsqueeze(-2) - position.unsqueeze(-3)).norm(dim=-1, keepdim=True) / self.grid_size
        self.ctx.set(rounds, edges, timed)

    @torch.no_grad()
    def act(self, obs, state, eval=False):
        "`Dreamer.act` with the graph of `obs`."
        self.set_graph(obs, timed=False)
        return super().act(obs, state, eval)

    def update(self, replay_buffer):
        "`Dreamer.update` with the graphs of the sampled batch."
        return super().update(_WithGraph(replay_buffer, self))


class _WithGraph:
    "A replay whose `sample` also sets the model's graph."
    def __init__(self, replay, model: CoDreamerModel):
        self.replay, self.model = replay, model

    def sample(self, *args, **kwargs):
        data, index, initial = self.replay.sample(*args, **kwargs)
        self.model.set_graph(data, timed=True)
        return data, index, initial

    def update(self, *args):
        return self.replay.update(*args)

### The agents

`CoDreamerAgents` is `DreamerAgents` with a `CoDreamerModel`: every transition also records `hears`
(each agent's incoming messages per round: the alive agents within `comm_range`, minus lost ones
with `drop_prob`, and with `links` what the comm-core networks delivered of each round's messages,
`message_bits`). Training (`train_dreamer`), acting and saving are `DreamerAgents`'.

In [ ]:
#| export
class CoDreamerAgents(DreamerAgents):
    "DreamerV3 agents that communicate as CoDreamer (graph attention over the agents within range)."
    def __init__(
        self,
        config: DictConfig, # See `dreamer_config`
        obs_dim: int,
        num_agents: int,
        num_actions: int = 4,
        obs_keys: Sequence[str] = ('image', 'direction'),
        goal: bool = True,
        grid_size: int = 15,
        team_reward: bool = False,
        comm_range: int | None = None, # The paper's C: agents hear the agents within this many cells (None: all)
        drop_prob: float = 0.0, # Probability that a message is lost (each pair and round)
        gnn_dim: int | None = None, # Width of the GATv2 layers (None: the MLPs' units)
    ):
        object.__setattr__(self, '_comm', dict(comm_range=comm_range, drop_prob=drop_prob, gnn_dim=gnn_dim))
        super().__init__(config, obs_dim, num_agents, num_actions, obs_keys, goal, grid_size, team_reward)
        self.comm_range, self.drop_prob, self.gnn_dim = comm_range, drop_prob, gnn_dim
        self.config.update(self._comm)
        self.links: NetworkLinks | None = None

    def _build(self, config, obs_space, act_space) -> CoDreamerModel:
        return CoDreamerModel(config, obs_space, act_space, self.num_agents, self._comm['gnn_dim'], self.grid_size)

    @property
    def message_bits(self) -> list[int]:
        "Bits of each round's message: the recurrent state, then the features (float32)."
        return [self.agent.rssm._deter * 32, self.agent.rssm.feat_size * 32]

    def on_reset(self, envs, seeds=None):
        if self.links is not None:
            self.links.reset(envs, seeds)

    def transitions(self, columns, reward, is_first, env_last, acting) -> dict[str, torch.Tensor]:
        "`DreamerAgents.transitions`, with `hears` (E, N, rounds, N)."
        trans = super().transitions(columns, reward, is_first, env_last, acting)
        alive = trans['mask'][..., 0] > 0
        position = np.asarray(columns['position'])
        hears = torch.stack([comm_mask(alive, torch.as_tensor(position) if self.comm_range is not None else None,
                                       self.comm_range, self.drop_prob) for _ in ROUNDS], 2)
        if self.links is not None:
            delivered = self.links.masks(position, alive.numpy(), len(ROUNDS), 0, message_bits=self.message_bits)
            hears = hears & delivered.transpose(1, 2)
        trans['hears'] = hears
        return trans

### Tests

A small CoDreamer on a small FindGoal: an agent's posterior and actor depend on another agent's
state only when they communicate; a short run trains (communication over comm-core included) and
acts in a `World`; the model loads back.

In [ ]:
#| hide
import tempfile
import comm_core as cc
from stable_marl import World
from stable_marl.policy import FeedForwardPolicy
from stable_marl.wm.commnet import commnet_features
from stable_marl.wm.dreamer import dreamer_config, train_dreamer

kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=10)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
world.reset(seed=[0, 1])
columns = {k: v[:, -1] for k, v in world.infos.items() if not k.startswith('_')}
dim = commnet_features(columns, grid_size=7).shape[-1]
small = dict(deter=128, hidden=32, units=32, discrete=8)
torch.manual_seed(0)
model = CoDreamerAgents(dreamer_config(**small), dim, 2, grid_size=7)
net = model.agent

# communication: agent 0's posterior and policy see agent 1's recurrent state and features, unless it does not hear it
trans = model.transitions(columns, np.zeros((2, 2)), np.ones(2, bool), np.zeros(2, bool), np.ones((2, 2), bool))
flat = {k: v.flatten(0, 1) for k, v in trans.items()}
for heard in (True, False):
    f = {**flat, 'hears': flat['hears'] & (torch.tensor(heard) | torch.eye(2, dtype=torch.bool).repeat(2, 1).unsqueeze(1))}
    net.set_graph(f, timed=False)
    deter = torch.randn(4, 128)
    other = deter.clone(); other[1::2] += 1                                   # agent 1 of each env
    with torch.no_grad():
        changed = not torch.allclose(net.rssm.prior(deter)[1][0::2], net.rssm.prior(other)[1][0::2])
        feat = torch.randn(4, net.rssm.feat_size); feat2 = feat.clone(); feat2[1::2] += 1
        changed_policy = not torch.allclose(net.actor(feat).logits[0::2], net.actor(feat2).logits[0::2])
    test_eq((changed, changed_policy), (heard, heard))
test_eq(model.message_bits, [128 * 32, net.rssm.feat_size * 32])
test_eq(trans['hears'].shape, (2, 2, 2, 2))

logs = []
train_dreamer(model, world, steps=60, batch_size=2, batch_length=8, train_ratio=4, log_every=30, log=logs.append)
assert logs and all(np.isfinite(r['loss/dyn']) for r in logs if 'loss/dyn' in r)
world.set_policy(FeedForwardPolicy(model))
test_eq(len(world.evaluate(episodes=2, seed=100)['episode_successes']), 2)

# over comm-core: each round's messages of their sizes; a channel losing everything leaves each agent alone
model.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=0.0))
world.evaluate(episodes=2, seed=100)
m = model.links.metrics[0]
assert m.transmissions > 0
test_eq(m.bits_sent, (m.transmissions // 2) * sum(model.message_bits))
model.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=1.0))
trans = model.transitions(columns, np.zeros((2, 2)), np.ones(2, bool), np.zeros(2, bool), np.ones((2, 2), bool))
test_eq(trans['hears'].any(), False)
train_dreamer(model, world, steps=40, batch_size=2, batch_length=8, train_ratio=2)
model.links = None

path = model.save(f'{tempfile.mkdtemp()}/codreamer.pt', env_name='MultiGrid-FindGoal-15x15-v0', env_kwargs=kw)
again = CoDreamerAgents.load(path)
test_eq((type(again.agent), again.checkpoint['env_kwargs']), (CoDreamerModel, kw))
for k, v in model.state_dict().items():
    test_eq(again.state_dict()[k], v)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()